# GeoAgent + leafmap (MapLibre) + Claude

- **`ANTHROPIC_API_KEY`** required.
- Install: `pip install "GeoAgent[anthropic,leafmap]"`.
- MapLibre camera state lives on **`m.view_state`** (not ipyleaflet). The `get_map_state` tool reads that structure.
- Optional **`ANTHROPIC_MODEL`** (default `claude-sonnet-4-6`).

In [ ]:
%pip install -q "GeoAgent[anthropic,leafmap]" leafmap

In [ ]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY")

MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-6")

In [ ]:
import leafmap.maplibregl as leafmap
from geoagent import for_leafmap
from geoagent.core.config import GeoAgentConfig
from geoagent.tools.leafmap import leafmap_tools

m = leafmap.Map(center=[-83.92, 35.96], zoom=9, height="520px")
m

## Map widget

In [ ]:
vs = getattr(m, "view_state", None)
if isinstance(vs, dict):
    print("view_state keys:", list(vs.keys()))
else:
    print("view_state:", type(vs).__name__ if vs is not None else None)

cfg = GeoAgentConfig(
    provider="anthropic",
    model=MODEL,
    temperature=0.0,
    max_tokens=4096,
)
agent = for_leafmap(m, config=cfg, fast=True)

get_state = next(t for t in leafmap_tools(m) if t.tool_name == "get_map_state")
st = get_state()
print("get_map_state zoom / center:", st.get("zoom"), st.get("center"))

resp = agent.chat("Using the map state, what is the zoom level? One sentence.")
print(resp.answer_text)
print("tools:", resp.executed_tools)

In [ ]:
resp = agent.chat("Fly to San Francisco")
print(resp.answer_text)
print("tools:", resp.executed_tools)